In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully! ✅")


Libraries imported successfully! ✅


In [2]:
df = pd.read_csv("../data/processed/manali_places_enriched.csv")
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (20, 20)


,place_id,name,address,country,latitude,longitude,rating,reviews,category,data_quality_score,nature,history,culture,adventure,photography,shopping,religious,family,travel_tags,interest_count
0,ChIJPeVowAaIBDkRnIfuPkskqi0,Hadimba Devi Temple,"Regency Road, Siyal Rd, Siyal, Manali, Himacha...",India,32.248353,77.181573,4.6,49688,Tourist attraction,6,1,1,1,0,1,1,1,0,"culture, history, nature, photography, religio...",6
1,ChIJ3VQyL0WHBDkR4Uml16nJto8,Old Manali snow point,"65XJ+J2W, Hadimba Temple Path, Old Manali, Man...",India,32.249112,77.180076,4.6,428,Tourist attraction,6,1,1,1,0,1,0,0,1,"culture, family, history, nature, photography",5
2,ChIJk9K3np2HBDkRS4FxsgYCMKI,Nehru Kund,"Bashisht, Himachal Pradesh 175103, India",India,32.285982,77.179824,4.4,7767,Tourist attraction,6,0,1,0,0,1,0,0,0,"history, photography",2
3,ChIJJSBNlWOJBDkRkmyIuy0OvGE,Kullu Manali River rafting,"65VQ+7MF, Siyal, Manali, Himachal Pradesh 1751...",India,32.243187,77.189176,4.5,88,Tourist attraction,6,0,0,0,0,0,0,0,0,NaN,0
4,ChIJHZ3eboyHBDkRBLrRpkcXmO4,Jogini Falls,"On water fall way V.P.O.-Vashist 5 km.from, Ma...",India,32.275076,77.188146,4.6,10842,Tourist attraction,6,1,1,0,0,1,0,0,1,"family, history, nature, photography",4


In [3]:
text_columns = ["name", "address", "category", "travel_tags"]

df["text"] = (
    df["name"].fillna("") + " "
    + df["address"].fillna("") + " "
    + df["category"].fillna("") + " "
    + df["travel_tags"].fillna("")
)

df[["name", "text"]].head()


,name,text
0,Hadimba Devi Temple,"Hadimba Devi Temple Regency Road, Siyal Rd, Si..."
1,Old Manali snow point,"Old Manali snow point 65XJ+J2W, Hadimba Temple..."
2,Nehru Kund,"Nehru Kund Bashisht, Himachal Pradesh 175103, ..."
3,Kullu Manali River rafting,"Kullu Manali River rafting 65VQ+7MF, Siyal, Ma..."
4,Jogini Falls,Jogini Falls On water fall way V.P.O.-Vashist ...


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)
df[["name", "clean_text"]].head()


,name,clean_text
0,Hadimba Devi Temple,hadimba devi temple regency road siyal rd si...
1,Old Manali snow point,old manali snow point 65xj j2w hadimba temple...
2,Nehru Kund,nehru kund bashisht himachal pradesh 175103 ...
3,Kullu Manali River rafting,kullu manali river rafting 65vq 7mf siyal ma...
4,Jogini Falls,jogini falls on water fall way v p o vashist ...


In [5]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (20, 259)


In [6]:
feature_names = vectorizer.get_feature_names_out()

print("Number of vocabulary terms:", len(feature_names))
print(feature_names[:50])


Number of vocabulary terms: 259
['175103' '175103 india' '175131' '175131 india' '175143' '175143 india'
 '5p8' '5p8 manali' '65mp' '65mp gw' '65pq' '65pq xc7' '65qq' '65qq jjw'
 '65vq' '65vq 7mf' '65vq hqw' '65wq' '65wq 6wm' '65xg' '65xg h26' '65xj'
 '65xj j2w' '6wm' '6wm siyal' '7539' '7539 w6c' '7mf' '7mf siyal' '859q'
 '859q 5p8' '86p9' '86p9 g8x' 'aleo' 'aleo manali' 'atal' 'atal bihari'
 'attraction' 'attraction culture' 'attraction family'
 'attraction history' 'attraction nature' 'attraction photography' 'baror'
 'baror manali' 'baror parsha' 'bashisht' 'bashisht himachal' 'bazaar'
 'bazaar 65vq']


In [7]:
def recommend_by_text(query, top_n=5):
    query_clean = clean_text(query)
    query_vector = vectorizer.transform([query_clean])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    result = df.copy()
    result["tfidf_similarity"] = similarity_scores

    return result.sort_values(
        "tfidf_similarity",
        ascending=False
    )[[
        "name", "category", "rating", "reviews",
        "travel_tags", "tfidf_similarity"
    ]].head(top_n).reset_index(drop=True)


In [8]:
query_1 = "peaceful natural places with beautiful photography opportunities"
recommend_by_text(query_1, top_n=5)


,name,category,rating,reviews,travel_tags,tfidf_similarity
0,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.197149
1,Manali View Point,Tourist attraction,4.6,87,photography,0.174314
2,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.143194
3,Van Vihar National Park,Tourist attraction,4.2,9050,"family, history, nature, photography",0.136012
4,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.131824


In [9]:
query_2 = "historical temples and cultural places"
recommend_by_text(query_2, top_n=5)


,name,category,rating,reviews,travel_tags,tfidf_similarity
0,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.0
1,Old Manali snow point,Tourist attraction,4.6,428,"culture, family, history, nature, photography",0.0
2,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.0
3,Kullu Manali River rafting,Tourist attraction,4.5,88,NaN,0.0
4,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.0


In [10]:
query_3 = "adventure activities and trekking"
recommend_by_text(query_3, top_n=5)


,name,category,rating,reviews,travel_tags,tfidf_similarity
0,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.0
1,Old Manali snow point,Tourist attraction,4.6,428,"culture, family, history, nature, photography",0.0
2,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.0
3,Kullu Manali River rafting,Tourist attraction,4.5,88,NaN,0.0
4,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.0


In [11]:
query_4 = "shopping and local tourist experiences"
recommend_by_text(query_4, top_n=5)


,name,category,rating,reviews,travel_tags,tfidf_similarity
0,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.195418
1,Manali Bazaar,Tourist attraction,4.3,3991,"family, history, nature, shopping",0.182026
2,Himachal PARDESH,Tourist attraction,3.9,11,NaN,0.033149
3,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.030197
4,Himalayan Igloo,Tourist attraction,4.5,199,NaN,0.028300


In [12]:
query_a = "peaceful natural photography"
query_b = "quiet scenic places for taking pictures"

print("Query A")
display(recommend_by_text(query_a, top_n=5))

print("Query B")
display(recommend_by_text(query_b, top_n=5))


Query A


,name,category,rating,reviews,travel_tags,tfidf_similarity
0,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.197149
1,Manali View Point,Tourist attraction,4.6,87,photography,0.174314
2,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.143194
3,Van Vihar National Park,Tourist attraction,4.2,9050,"family, history, nature, photography",0.136012
4,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.131824


Query B


,name,category,rating,reviews,travel_tags,tfidf_similarity
0,Hadimba Devi Temple,Tourist attraction,4.6,49688,"culture, history, nature, photography, religio...",0.0
1,Old Manali snow point,Tourist attraction,4.6,428,"culture, family, history, nature, photography",0.0
2,Nehru Kund,Tourist attraction,4.4,7767,"history, photography",0.0
3,Kullu Manali River rafting,Tourist attraction,4.5,88,NaN,0.0
4,Jogini Falls,Tourist attraction,4.6,10842,"family, history, nature, photography",0.0


In [13]:
rating_min = df["rating"].min()
rating_max = df["rating"].max()

if rating_max == rating_min:
    df["rating_normalized"] = 1.0
else:
    df["rating_normalized"] = (
        (df["rating"] - rating_min)
        / (rating_max - rating_min)
    )

def recommend_by_text_hybrid(query, top_n=5):
    query_clean = clean_text(query)
    query_vector = vectorizer.transform([query_clean])

    similarity_scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    result = df.copy()
    result["tfidf_similarity"] = similarity_scores
    result["hybrid_score"] = (
        0.80 * result["tfidf_similarity"]
        + 0.20 * result["rating_normalized"]
    )

    return result.sort_values(
        "hybrid_score",
        ascending=False
    )[[
        "name", "rating", "reviews", "travel_tags",
        "tfidf_similarity", "hybrid_score"
    ]].head(top_n).reset_index(drop=True)


In [14]:
recommend_by_text_hybrid(
    "peaceful natural places with beautiful photography",
    top_n=5
)


,name,rating,reviews,travel_tags,tfidf_similarity,hybrid_score
0,Manali View Point,4.6,87,photography,0.174314,0.295007
1,Jogini Falls,4.6,10842,"family, history, nature, photography",0.143194,0.270111
2,Nehru Kund,4.4,7767,"history, photography",0.197149,0.268830
3,Hadimba Devi Temple,4.6,49688,"culture, history, nature, photography, religio...",0.131824,0.261015
4,Old Manali snow point,4.6,428,"culture, family, history, nature, photography",0.128360,0.258243


In [15]:
query = "peaceful natural places with beautiful photography"

query_clean = clean_text(query)

print("Cleaned query:")
print(query_clean)

print("\nWords in query:")
print(query_clean.split())

Cleaned query:
peaceful natural places with beautiful photography

Words in query:
['peaceful', 'natural', 'places', 'with', 'beautiful', 'photography']


In [16]:
query_vector = vectorizer.transform([query_clean])

print("Number of non-zero query terms:",
      query_vector.nnz)

Number of non-zero query terms: 1


In [17]:
query_words = query_clean.split()

for word in query_words:
    if word in vectorizer.vocabulary_:
        print(f"{word} ✅")
    else:
        print(f"{word} ❌")

peaceful ❌
natural ❌
places ❌
with ❌
beautiful ❌
photography ✅


In [18]:
df["recommendation_text"] = (
    df["name"].fillna("") + " "
    + df["category"].fillna("") + " "
    + df["travel_tags"].fillna("")
)

df["recommendation_text"] = df["recommendation_text"].apply(clean_text)

In [19]:
better_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

better_tfidf_matrix = better_vectorizer.fit_transform(
    df["recommendation_text"]
)

In [20]:
def recommend_by_better_text(query, top_n=5):

    query_clean = clean_text(query)

    query_vector = better_vectorizer.transform(
        [query_clean]
    )

    similarity = cosine_similarity(
        query_vector,
        better_tfidf_matrix
    ).flatten()

    result = df.copy()
    result["similarity"] = similarity

    return result.sort_values(
        "similarity",
        ascending=False
    )[[
        "name",
        "travel_tags",
        "similarity"
    ]].head(top_n)

In [21]:
recommend_by_better_text(
    "peaceful natural places with beautiful photography",
    5
)

,name,travel_tags,similarity
6,Manali View Point,photography,0.268844
2,Nehru Kund,"history, photography",0.249741
4,Jogini Falls,"family, history, nature, photography",0.238977
5,Van Vihar National Park,"family, history, nature, photography",0.189965
1,Old Manali snow point,"culture, family, history, nature, photography",0.188284
